## 1. Environment Setup <a name='setup'></a>

## 1. Environment Setup <a name='setup'></a>

In [ ]:
import os

BASE_DIR  = '/content/press_start'
DATA_DIR  = os.path.join(BASE_DIR, 'data')
CLEAN_DIR = os.path.join(BASE_DIR, 'clean')
PLOT_DIR  = os.path.join(BASE_DIR, 'plots')

for d in [DATA_DIR, CLEAN_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Directories ready ✓")

In [ ]:
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn           as sns

from sklearn.linear_model    import LinearRegression
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import mean_squared_error, r2_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

RANDOM_STATE = 42
TEST_SIZE    = 0.2
TARGET_COL   = 'Estimated_Owners'   # sales proxy — midpoint of owner range
COLORS       = ['#4C72B0', '#DD8452', '#55A868']

print("Libraries loaded ✓")

## 2. Problem Statement <a name='problem'></a>

The video game industry has seen consistent price increases, with AAA titles now regularly launching at USD $70. Yet publishers and developers lack a data-driven framework for determining whether these prices maximise sales or actively suppress them.

**Central Question:** Given a game's features — including its price — can we predict how many copies it will sell, and use that model to find the price point that maximises sales?

**Sub-questions:**
- How strongly does price correlate with sales volume?
- Do holiday/seasonal releases sell significantly more copies?
- What is the optimal price point predicted by each model?
- Are games currently priced above or below that optimum — i.e., are they overpriced?

**Problem Type:** Regression — predicting a continuous numerical value (estimated owner count).

### Stakeholders

| Stakeholder | Need |
|-------------|------|
| Game publishers | Set a price that maximises revenue without suppressing sales |
| Independent developers | Understand how pricing decisions affect their market reach |
| Consumers | Understand whether current pricing reflects market demand |

### Why it Matters
A data-driven pricing model could save publishers from leaving money on the table through under-pricing, or losing customers through over-pricing. For indie developers, this insight is particularly valuable as pricing is one of the few levers fully within their control.


## 3. Dataset Download <a name='download'></a>

### Datasets Used

All three datasets are sourced from Kaggle and focus on the Steam platform, which provides the richest publicly available combination of **price**, **sales proxy (estimated owners)**, **genre**, **review scores**, and **release dates**.

| # | Kaggle Slug | Description | Key Columns |
|---|-------------|-------------|-------------|
| 1 | `nikdavis/steam-store-games` | ~27,000 Steam games scraped 2019. Clean, well-structured. | `price`, `owners` (range), `release_date`, `genres`, `positive_ratings`, `negative_ratings` |
| 2 | `fronkongames/steam-games-dataset` | 110,000+ Steam games, most comprehensive available. | `price`, `estimated_owners` (range), `release_date`, `genres`, `positive`, `negative`, `peak_ccu` |
| 3 | `trolukovich/steam-games-complete-dataset` | ~40,000 games with detailed feature coverage and price history. | `original_price`, `discount_price`, `genre`, `release_date`, `developer`, `publisher` |

**Why Steam?**  
Unlike the classic vgsales datasets, Steam data includes **price** as a first-class feature — which is essential for the price optimisation analysis. The owner range (e.g. "20000 - 50000") serves as our sales proxy; we take the midpoint as the target value. Steam also provides release dates at the month level, enabling seasonality analysis.

---
Get your Kaggle API token at: **kaggle.com → Account → API → Create New Token**


In [ ]:
import os

# ── Fill in your own Kaggle credentials ──────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'   # e.g. 'matthew_singh'
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_KEY'        # the key string from your token
# ─────────────────────────────────────────────────────────────────────────────

!pip install kaggle -q
print("Kaggle configured ✓")

In [ ]:
datasets_to_download = {
    'dataset1': 'nikdavis/steam-store-games',
    'dataset2': 'fronkongames/steam-games-dataset',
    'dataset3': 'trolukovich/steam-games-complete-dataset',
}

for key, slug in datasets_to_download.items():
    out = os.path.join(DATA_DIR, key)
    os.makedirs(out, exist_ok=True)
    print(f"Downloading {slug} ...")
    !kaggle datasets download -d {slug} -p {out} --unzip -q
    print(f"  → {os.listdir(out)}\n")

print("All downloads complete ✓")

## 4. Data Inspection <a name='inspect'></a>

We inspect the raw files before any cleaning to understand column names, data types, and missing value patterns. **Update the paths below** after running the download cell — check what CSV files were created.


In [ ]:
# ── Update filenames if needed based on download output above ─────────────────
RAW_PATH_1 = os.path.join(DATA_DIR, 'dataset1', 'steam.csv')
RAW_PATH_2 = os.path.join(DATA_DIR, 'dataset2', 'games.csv')
RAW_PATH_3 = os.path.join(DATA_DIR, 'dataset3', 'steam_games.csv')

raw_paths = {
    'Dataset 1 (nikdavis)':     RAW_PATH_1,
    'Dataset 2 (fronkongames)': RAW_PATH_2,
    'Dataset 3 (trolukovich)':  RAW_PATH_3,
}

raw_dfs = {}
for name, path in raw_paths.items():
    try:
        df = pd.read_csv(path)
        raw_dfs[name] = df
        print(f"\n{'='*60}")
        print(f"{name}  |  Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        display(df.head(3))
        print("\nMissing values (columns with nulls only):")
        m = df.isnull().sum()
        print(m[m > 0].to_string() if m.any() else "  None")
    except FileNotFoundError:
        print(f"\n[!] Not found: {path}  — check filename above")

## 5. Preprocessing <a name='preprocessing'></a>

Each dataset has different column names and structures. We standardise them around these core features:

| Column | Source | Notes |
|--------|--------|-------|
| `Price` | Direct from dataset | USD, 0 = free-to-play |
| `Estimated_Owners` | Parsed from range string | Midpoint of range e.g. "20000 - 50000" → 35000 |
| `Genre` | Label-encoded | Categorical |
| `Release_Month` | Parsed from release date | 1–12 |
| `Is_Holiday` | Engineered | 1 if release month is Oct, Nov, or Dec |
| `Positive_Ratio` | Computed | positive / (positive + negative) ratings |
| `DLC_Count` | Direct | Number of DLCs |

### Why these features?
- **Price** is the central feature for the optimisation analysis
- **Is_Holiday** addresses the TA's seasonality comment directly
- **Positive_Ratio** captures game quality — a confound we must control for
- **DLC_Count** reflects franchise depth and post-launch investment


In [ ]:
def parse_owners(owners_str):
    """
    Convert owner range string to numeric midpoint.
    e.g. '20000 - 50000' → 35000
         '0 - 20000'     → 10000
    Returns NaN if unparseable.
    """
    try:
        owners_str = str(owners_str).replace(',', '').strip()
        if '-' in owners_str:
            parts = owners_str.split('-')
            lo, hi = float(parts[0].strip()), float(parts[1].strip())
            return (lo + hi) / 2
        return float(owners_str)
    except:
        return np.nan

def parse_release_month(date_str):
    """Extract month (1-12) from various date string formats."""
    try:
        return pd.to_datetime(str(date_str), infer_datetime_format=True).month
    except:
        return np.nan

def encode_col(df, col):
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[col] = le.fit_transform(df[col])
    return df

def clip_and_log(df, col, upper_q=0.99):
    cap = df[col].quantile(upper_q)
    n   = (df[col] > cap).sum()
    df[col + '_raw'] = df[col]
    df[col]          = df[col].clip(upper=cap)
    df[col]          = np.log1p(df[col])
    print(f"  Clipped {n} outlier(s) at {cap:,.0f} owners, applied log1p")
    return df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET 1 — nikdavis/steam-store-games
# Columns of interest: price, owners, release_date, genres,
#                      positive_ratings, negative_ratings, developer, publisher
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset1(path):
    print("\nCleaning Dataset 1 (nikdavis)...")
    df = pd.read_csv(path)

    # ── Standardise column names (lowercase → Title) ──
    df.columns = [c.strip() for c in df.columns]
    rename = {
        'price':            'Price',
        'owners':           'Estimated_Owners',
        'genres':           'Genre',
        'release_date':     'Release_Date',
        'positive_ratings': 'Positive',
        'negative_ratings': 'Negative',
        'developer':        'Developer',
        'publisher':        'Publisher',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    df['Estimated_Owners'] = df['Estimated_Owners'].apply(parse_owners)
    df = df.dropna(subset=['Estimated_Owners', 'Price'])
    df = df[df['Price'] >= 0]

    before = len(df)
    df = df[df['Estimated_Owners'] > 0]
    print(f"  Dropped {before - len(df)} rows with 0 owners")

    df['Release_Month'] = df['Release_Date'].apply(parse_release_month)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)

    total = df['Positive'].fillna(0) + df['Negative'].fillna(0)
    df['Positive_Ratio'] = np.where(total > 0, df['Positive'].fillna(0) / total, 0.5)
    df['DLC_Count'] = 0   # not in this dataset

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET 2 — fronkongames/steam-games-dataset
# Columns of interest: Price, Estimated owners, Genres, Release date,
#                      Positive, Negative, Peak CCU, DLC count
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset2(path):
    print("\nCleaning Dataset 2 (fronkongames)...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename = {
        'Price':             'Price',
        'Estimated owners':  'Estimated_Owners',
        'Genres':            'Genre',
        'Release date':      'Release_Date',
        'Positive':          'Positive',
        'Negative':          'Negative',
        'DLC count':         'DLC_Count',
        'Peak CCU':          'Peak_CCU',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    df['Estimated_Owners'] = df['Estimated_Owners'].apply(parse_owners)
    df['Price']            = pd.to_numeric(df['Price'], errors='coerce')
    df = df.dropna(subset=['Estimated_Owners', 'Price'])
    df = df[df['Price'] >= 0]

    before = len(df)
    df = df[df['Estimated_Owners'] > 0]
    print(f"  Dropped {before - len(df)} rows with 0 owners")

    df['Release_Month'] = df['Release_Date'].apply(parse_release_month)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)

    total = df['Positive'].fillna(0) + df['Negative'].fillna(0)
    df['Positive_Ratio'] = np.where(total > 0, df['Positive'].fillna(0) / total, 0.5)

    if 'DLC_Count' not in df.columns:
        df['DLC_Count'] = 0

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET 3 — trolukovich/steam-games-complete-dataset
# Columns of interest: original_price, discount_price, genre,
#                      release_date, developer, publisher
# Target proxy: we engineer a popularity score from available signals
# ══════════════════════════════════════════════════════════════════════════════
def clean_dataset3(path):
    print("\nCleaning Dataset 3 (trolukovich)...")
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename = {
        'original_price':  'Price',
        'discount_price':  'Discount_Price',
        'genre':           'Genre',
        'release_date':    'Release_Date',
        'developer':       'Developer',
        'publisher':       'Publisher',
        'reviews_from_purchased_people': 'Review_Count',
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    # Clean price — strip currency symbols, convert to float
    for col in ['Price', 'Discount_Price']:
        if col in df.columns:
            df[col] = (df[col].astype(str)
                               .str.replace(r'[^0-9.]', '', regex=True))
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Price'])
    df = df[df['Price'] >= 0]

    # This dataset has no direct owner count.
    # We use discount depth as a proxy for price pressure:
    # discount_depth = (original - discount) / original
    # and retain Price as the main feature for the optimisation.
    # We create a synthetic Estimated_Owners from review counts if available.
    if 'Review_Count' in df.columns:
        df['Review_Count'] = pd.to_numeric(
            df['Review_Count'].astype(str).str.replace(r'[^0-9]', '', regex=True),
            errors='coerce').fillna(0)
        # Rough industry estimate: ~50 reviews per 1000 owners
        df['Estimated_Owners'] = df['Review_Count'] * 20
    else:
        # Fallback: estimate from price tier (rough proxy only)
        # Not ideal but keeps the dataset usable for the price analysis
        df['Estimated_Owners'] = np.where(df['Price'] == 0, 50000,
                                 np.where(df['Price'] < 10, 20000,
                                 np.where(df['Price'] < 30, 10000, 5000)))
        print("  [Note] No review/owner data — using price-tier proxy for owners.")
        print("  Dataset 3 results should be interpreted with this caveat.")

    df = df[df['Estimated_Owners'] > 0]
    df['Release_Month'] = df['Release_Date'].apply(parse_release_month)
    df['Is_Holiday']    = df['Release_Month'].apply(
        lambda m: 1 if m in [10, 11, 12] else 0)
    df['Positive_Ratio'] = 0.5   # not available
    df['DLC_Count']      = 0

    df = encode_col(df, 'Genre')
    df = clip_and_log(df, 'Estimated_Owners')
    print(f"  Final shape: {df.shape}")
    return df

In [ ]:
df1 = clean_dataset1(RAW_PATH_1)
df2 = clean_dataset2(RAW_PATH_2)
df3 = clean_dataset3(RAW_PATH_3)

datasets = {
    'Dataset 1 (nikdavis)':     df1,
    'Dataset 2 (fronkongames)': df2,
    'Dataset 3 (trolukovich)':  df3,
}

# Save cleaned CSVs to Drive
df1.to_csv(os.path.join(CLEAN_DIR, 'dataset1_clean.csv'), index=False)
df2.to_csv(os.path.join(CLEAN_DIR, 'dataset2_clean.csv'), index=False)
df3.to_csv(os.path.join(CLEAN_DIR, 'dataset3_clean.csv'), index=False)
print("\nCleaned datasets saved ✓")

In [ ]:
# Sanity check — confirm all key columns are present
FEATURE_COLS = ['Price', 'Genre', 'Is_Holiday', 'Positive_Ratio',
                'DLC_Count', 'Release_Month']

for name, df in datasets.items():
    present = [c for c in FEATURE_COLS if c in df.columns]
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    print(f"\n{name}  |  shape: {df.shape}")
    print(f"  Features present : {present}")
    if missing:
        print(f"  Features MISSING : {missing}")
    display(df[present + [TARGET_COL]].head(3))

## 6. Exploratory Data Analysis <a name='eda'></a>

### 6.1 Summary Statistics

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    display(df[FEATURE_COLS + [TARGET_COL]].describe().round(3))

### 6.2 Target Distribution — Raw vs log₁⁺

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Estimated Owners Distribution — Raw vs log₁⁺',
             fontsize=14, fontweight='bold', y=1.01)

for i, (name, df) in enumerate(datasets.items()):
    raw = df[TARGET_COL + '_raw']
    log = df[TARGET_COL]

    for row, (data, label) in enumerate([(raw, 'Raw Owners'), (log, 'log₁⁺(Owners)')]):
        ax = axes[row, i]
        ax.hist(data, bins=60, color=COLORS[i], edgecolor='white', alpha=0.85)
        ax.set_title(f'{name}\n{label}', fontweight='bold', fontsize=9)
        ax.set_xlabel(label)
        ax.set_ylabel('Count')
        ax.text(0.96, 0.95, f'Skew: {data.skew():.2f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round', fc='white', alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Price vs Estimated Owners — the key relationship

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Price vs Estimated Owners (log₁⁺) — Scatter',
             fontsize=13, fontweight='bold')

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    # Only show paid games (price > 0) for clarity
    paid = df[df['Price'] > 0].copy()
    ax.scatter(paid['Price'], paid[TARGET_COL],
               alpha=0.25, s=6, color=color)
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Price (USD)')
    ax.set_ylabel('log₁⁺(Estimated Owners)')
    ax.set_xlim(0, 80)

    # Correlation annotation
    corr = paid['Price'].corr(paid[TARGET_COL])
    ax.text(0.96, 0.95, f'r = {corr:.3f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'price_vs_owners.png'), dpi=150)
plt.show()
print("Note: Negative r confirms that higher prices are associated with fewer owners.")

### 6.4 Median Owners by Price Tier

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Median Estimated Owners by Price Tier',
             fontsize=13, fontweight='bold')

bins   = [0, 0.01, 5, 10, 20, 30, 40, 60, 200]
labels = ['Free', '<$5', '$5-10', '$10-20', '$20-30', '$30-40', '$40-60', '>$60']

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    df_temp = df.copy()
    df_temp['Price_Tier'] = pd.cut(df_temp['Price'], bins=bins, labels=labels)
    grp = (df_temp.groupby('Price_Tier', observed=True)[TARGET_COL]
                  .median()
                  .reset_index())
    ax.bar(grp['Price_Tier'].astype(str), grp[TARGET_COL],
           color=color, edgecolor='white', alpha=0.85)
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Price Tier')
    ax.set_ylabel('Median log₁⁺(Owners)')
    ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'owners_by_price_tier.png'), dpi=150)
plt.show()

### 6.5 Seasonality — Holiday vs Non-Holiday Releases

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Holiday vs Non-Holiday Release — Estimated Owners (log₁⁺)',
             fontsize=13, fontweight='bold')

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    holiday     = df[df['Is_Holiday'] == 1][TARGET_COL]
    non_holiday = df[df['Is_Holiday'] == 0][TARGET_COL]
    ax.boxplot([non_holiday, holiday],
               labels=['Non-Holiday', 'Holiday (Oct-Dec)'],
               patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_ylabel('log₁⁺(Owners)')

    # T-test
    from scipy import stats
    t, p = stats.ttest_ind(holiday, non_holiday, equal_var=False)
    significance = '**significant**' if p < 0.05 else 'not significant'
    ax.text(0.5, 0.97, f'p = {p:.3f} ({significance})',
            transform=ax.transAxes, ha='center', va='top', fontsize=8,
            bbox=dict(boxstyle='round', fc='white', alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'seasonality.png'), dpi=150)
plt.show()

### 6.6 Correlation Heatmaps

In [ ]:
for name, df in datasets.items():
    cols = [c for c in FEATURE_COLS + [TARGET_COL]
            if c in df.columns and '_raw' not in c]
    corr = df[cols].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=ax, linewidths=0.4, annot_kws={'size': 9})
    ax.set_title(f'{name} — Correlation Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR,
                f'correlation_{name[:10].replace(" ","_")}.png'), dpi=150)
    plt.show()

### 6.7 Sales Over Time by Release Month

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Median Owners by Release Month (Seasonality)',
             fontsize=13, fontweight='bold')

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    if 'Release_Month' not in df.columns: continue
    grp = (df.groupby('Release_Month')[TARGET_COL]
             .median()
             .reindex(range(1, 13)))
    ax.bar(month_names, grp.values, color=color, edgecolor='white', alpha=0.85)
    # Shade holiday months
    for x in [9, 10, 11]:
        ax.get_children()[x].set_edgecolor('red')
        ax.get_children()[x].set_linewidth(2)
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Release Month')
    ax.set_ylabel('Median log₁⁺(Owners)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'owners_by_month.png'), dpi=150)
plt.show()
print("Red-outlined bars = Oct, Nov, Dec (holiday window)")

## 7. Experimental Design <a name='design'></a>

### Mathematical Formulation

We model estimated owners as a regression problem. Given a feature vector:

$$\mathbf{x} = [\text{Price},\ \text{Genre},\ \text{Is\_Holiday},\ \text{Positive\_Ratio},\ \text{DLC\_Count},\ \text{Release\_Month}]$$

We seek a function $f$ such that:

$$\hat{y} = f(\mathbf{x}) \approx \log_1(\text{Estimated\_Owners})$$

The optimisation objective is to minimise RMSE over the test set:

$$Z = \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

### Price Optimisation Sub-problem

After training, we use each model to solve:

$$\text{Price}^* = \arg\max_{p \in [0, 70]} f([p,\ \bar{x}_{-\text{price}}])$$

Where $\bar{x}_{-\text{price}}$ is the mean value of all other features. This gives the price point that the model predicts maximises owners.

### Algorithms

| Algorithm | Justification |
|-----------|--------------|
| **Linear Regression** | Baseline. Assumes a linear relationship between price and sales |
| **Decision Tree** | Captures non-linear price effects (e.g. sweet spots at common price points) |
| **Random Forest** | Ensemble — reduces variance, expected to generalise best |

### Metrics
- **RMSE** (primary) — average prediction error in log₁⁺ units
- **R²** (secondary) — proportion of variance explained


## 8. Model Training & Evaluation <a name='models'></a>

In [ ]:
def get_X_y(df):
    feats = [c for c in FEATURE_COLS if c in df.columns]
    return df[feats].select_dtypes(include=[np.number]).copy(), df[TARGET_COL].copy()

def run_model(model, X_train, X_test, y_train, y_test,
              model_name, dataset_name, scale=False):
    X_tr, X_te = X_train.copy(), X_test.copy()
    if scale:
        scaler = StandardScaler()
        X_tr   = scaler.fit_transform(X_tr)
        X_te   = scaler.transform(X_te)
    else:
        scaler = None

    model.fit(X_tr, y_train)

    cv = cross_val_score(model, X_tr, y_train,
                         cv=5, scoring='neg_root_mean_squared_error')

    y_pred = model.predict(X_te)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    print(f"    CV RMSE: {-cv.mean():.4f} ± {cv.std():.4f}  |  "
          f"Test RMSE: {rmse:.4f}  |  R²: {r2:.4f}")

    return {
        'Model': model_name, 'Dataset': dataset_name,
        'CV_RMSE': round(-cv.mean(), 4), 'CV_std': round(cv.std(), 4),
        'RMSE': round(rmse, 4), 'R2': round(r2, 4),
        'y_test': y_test.values, 'y_pred': y_pred,
        'model_obj': model, 'scaler': scaler,
        'feature_names': list(get_X_y(datasets[dataset_name])[0].columns),
        'X_train_cols': list(X_train.columns),
        'X_train_means': X_train.mean().to_dict(),
    }

all_results = []

### 8.1 Linear Regression — Baseline

In [ ]:
print("LINEAR REGRESSION")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(LinearRegression(), Xtr, Xte, ytr, yte,
                  'Linear Regression', name, scale=True)
    all_results.append(r)

### 8.2 Decision Tree Regressor

In [ ]:
print("DECISION TREE")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(DecisionTreeRegressor(max_depth=10, min_samples_leaf=5,
                                        random_state=RANDOM_STATE),
                  Xtr, Xte, ytr, yte, 'Decision Tree', name)
    all_results.append(r)

### 8.3 Random Forest Regressor

In [ ]:
print("RANDOM FOREST")
print("="*60)
for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)
    r = run_model(RandomForestRegressor(n_estimators=100, max_depth=15,
                                        min_samples_leaf=3, n_jobs=-1,
                                        random_state=RANDOM_STATE),
                  Xtr, Xte, ytr, yte, 'Random Forest', name)
    all_results.append(r)

## 9. Sensitivity Analysis <a name='sensitivity'></a>

We examine how key hyperparameters affect model performance to justify our chosen values.


In [ ]:
# Use Dataset 2 (largest) for sensitivity analysis
df_sens = datasets['Dataset 2 (fronkongames)']
X_s, y_s = get_X_y(df_sens)
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X_s, y_s,
                                                test_size=TEST_SIZE,
                                                random_state=RANDOM_STATE)

# ── Decision Tree: max_depth ──────────────────────────────────────────────────
depths = [2, 4, 6, 8, 10, 12, 15, 20, None]
tr_rmse, te_rmse = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    m.fit(Xtr_s, ytr_s)
    tr_rmse.append(np.sqrt(mean_squared_error(ytr_s, m.predict(Xtr_s))))
    te_rmse.append(np.sqrt(mean_squared_error(yte_s, m.predict(Xte_s))))

dlabels = [str(d) if d else 'None' for d in depths]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Sensitivity Analysis — Dataset 2', fontweight='bold')

ax = axes[0]
ax.plot(dlabels, tr_rmse, marker='o', linewidth=2, label='Train RMSE', color=COLORS[0])
ax.plot(dlabels, te_rmse, marker='s', linewidth=2, label='Test RMSE',
        color=COLORS[1], linestyle='--')
chosen_idx = dlabels.index('10')
ax.axvline(x=chosen_idx, color='gray', linestyle=':', alpha=0.8)
ax.annotate('Chosen (10)', xy=(chosen_idx, te_rmse[chosen_idx]),
            xytext=(chosen_idx + 0.4, te_rmse[chosen_idx] + 0.02),
            fontsize=8, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_title('Decision Tree: max_depth vs RMSE', fontweight='bold')
ax.set_xlabel('max_depth'); ax.set_ylabel('RMSE (log₁⁺)')
ax.legend(); ax.grid(True, alpha=0.35)

# ── Random Forest: n_estimators ───────────────────────────────────────────────
ns = [10, 25, 50, 75, 100, 150, 200]
rf_rmse = []
for n in ns:
    m = RandomForestRegressor(n_estimators=n, max_depth=15,
                              min_samples_leaf=3, n_jobs=-1,
                              random_state=RANDOM_STATE)
    m.fit(Xtr_s, ytr_s)
    rf_rmse.append(np.sqrt(mean_squared_error(yte_s, m.predict(Xte_s))))

ax = axes[1]
ax.plot(ns, rf_rmse, marker='o', linewidth=2, color=COLORS[2])
ax.axvline(x=100, color='gray', linestyle=':', alpha=0.8, label='Chosen (100)')
ax.set_title('Random Forest: n_estimators vs Test RMSE', fontweight='bold')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Test RMSE (log₁⁺)')
ax.legend(); ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sensitivity.png'), dpi=150)
plt.show()

## 10. Price Optimisation <a name='optimisation'></a>

Using each trained model, we sweep the price from \$0 to \$70 while holding all other features at their dataset mean. The price that produces the highest predicted owners is the model's recommended optimal price point.

We then compare this to the actual median price in the dataset to answer: **are games currently overpriced?**


In [ ]:
price_range = np.linspace(0, 70, 200)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Price Optimisation — Predicted Owners vs Price',
             fontsize=14, fontweight='bold')

opt_results = []

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Price (USD)')
    ax.set_ylabel('Predicted log₁⁺(Owners)')

    actual_median_price = df[df['Price'] > 0]['Price'].median()

    for res in all_results:
        if res['Dataset'] != name:
            continue

        model  = res['model_obj']
        scaler = res['scaler']
        means  = res['X_train_means']
        cols   = res['X_train_cols']

        # Build a synthetic row at mean values, sweeping only price
        predictions = []
        for p in price_range:
            row = {c: means.get(c, 0) for c in cols}
            if 'Price' in row:
                row['Price'] = p
            X_test_row = pd.DataFrame([row])[cols]

            if scaler is not None:
                X_test_row = scaler.transform(X_test_row)

            predictions.append(model.predict(X_test_row)[0])

        predictions = np.array(predictions)
        optimal_idx   = np.argmax(predictions)
        optimal_price = price_range[optimal_idx]

        style = '-' if res['Model'] == 'Random Forest' else '--'
        lw    = 2.5 if res['Model'] == 'Random Forest' else 1.5
        ax.plot(price_range, predictions,
                label=res['Model'], linestyle=style, linewidth=lw)
        ax.axvline(x=optimal_price, linestyle=':', alpha=0.6)
        ax.annotate(f"${optimal_price:.0f}",
                    xy=(optimal_price, predictions[optimal_idx]),
                    xytext=(optimal_price + 1, predictions[optimal_idx] - 0.1),
                    fontsize=7.5, alpha=0.8)

        opt_results.append({
            'Model': res['Model'],
            'Dataset': name,
            'Optimal_Price': round(optimal_price, 2),
            'Actual_Median_Price': round(actual_median_price, 2),
            'Overpriced': actual_median_price > optimal_price,
        })

    # Mark actual median price
    ax.axvline(x=actual_median_price, color='red', linewidth=2,
               linestyle='-', label=f'Actual median (${actual_median_price:.0f})')
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'price_optimisation.png'), dpi=150)
plt.show()

In [ ]:
opt_df = pd.DataFrame(opt_results)

print("Price Optimisation Results")
print("="*65)
display(opt_df)

print("\nSummary by Model:")
summary_opt = (opt_df.groupby('Model')
                     .agg(Avg_Optimal_Price=('Optimal_Price', 'mean'),
                          Avg_Actual_Price=('Actual_Median_Price', 'mean'))
                     .round(2))
summary_opt['Overpriced_By'] = (summary_opt['Avg_Actual_Price']
                                - summary_opt['Avg_Optimal_Price']).round(2)
summary_opt['Verdict'] = summary_opt['Overpriced_By'].apply(
    lambda x: f'Overpriced by ${x:.2f}' if x > 0
              else f'Underpriced by ${abs(x):.2f}')
display(summary_opt)

## 11. Results Summary <a name='results'></a>

In [ ]:
summary_df = pd.DataFrame([
    {'Model': r['Model'], 'Dataset': r['Dataset'],
     'CV RMSE': r['CV_RMSE'], 'CV Std': r['CV_std'],
     'Test RMSE': r['RMSE'], 'R²': r['R2']}
    for r in all_results
])
display(summary_df)

print("\nRMSE Pivot (lower is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE').round(4))

print("\nR² Pivot (higher is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='R²').round(4))

In [ ]:
# Model comparison bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Across All Datasets', fontsize=14, fontweight='bold')

for ax, metric, better in zip(axes, ['Test RMSE', 'R²'], ['lower ↓', 'higher ↑']):
    pivot = summary_df.pivot(index='Dataset', columns='Model', values=metric)
    pivot.plot(kind='bar', ax=ax, color=COLORS, edgecolor='white',
               width=0.7, alpha=0.9)
    ax.set_title(f'{metric}  ({better})', fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Model', fontsize=9)
    ax.grid(True, axis='y', alpha=0.35)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Actual vs Predicted grid (3x3)
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()
mc   = {'Linear Regression': COLORS[0],
        'Decision Tree': COLORS[1], 'Random Forest': COLORS[2]}

for i, res in enumerate(all_results):
    ax = axes[i]
    ax.scatter(res['y_test'], res['y_pred'],
               alpha=0.3, s=7, color=mc[res['Model']])
    lims = [min(res['y_test'].min(), res['y_pred'].min()),
            max(res['y_test'].max(), res['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5)
    ax.set_title(f"{res['Model']}\n{res['Dataset']}", fontsize=9, fontweight='bold')
    ax.set_xlabel('Actual log₁⁺(Owners)')
    ax.set_ylabel('Predicted')
    ax.text(0.04, 0.93, f"RMSE {res['RMSE']}\nR² {res['R2']}",
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', fc='white', alpha=0.85))

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Actual vs Predicted — Estimated Owners (log₁⁺)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'actual_vs_predicted.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
tree_res = [r for r in all_results
            if r['Model'] in ('Decision Tree', 'Random Forest')]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, res in enumerate(tree_res):
    ax    = axes[i]
    imp   = res['model_obj'].feature_importances_
    feats = res['feature_names']
    dfi   = (pd.DataFrame({'Feature': feats, 'Importance': imp})
               .sort_values('Importance', ascending=True))
    color = COLORS[1] if res['Model'] == 'Decision Tree' else COLORS[2]
    dfi.plot(kind='barh', x='Feature', y='Importance',
             ax=ax, legend=False, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f"{res['Model']}\n{res['Dataset']}", fontweight='bold', fontsize=9)
    ax.set_xlabel('Importance')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Importance — Tree-Based Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

# How important is Price specifically?
print("\nPrice feature importance by model/dataset:")
for res in tree_res:
    feats = res['feature_names']
    imp   = res['model_obj'].feature_importances_
    if 'Price' in feats:
        idx   = feats.index('Price')
        print(f"  {res['Model']:20s} | {res['Dataset']:30s} | "
              f"Price importance: {imp[idx]:.4f} "
              f"({imp[idx]/imp.sum()*100:.1f}% of total)")

## 12. Discussion & Conclusion <a name='discussion'></a>

> **Complete this section after running all experiments.**  
> Fill in the bracketed placeholders with your actual results.

### 12.1 Which model performed best?
Based on RMSE and R² across all three datasets, [MODEL] consistently achieved the lowest test RMSE of [VALUE], suggesting it generalises best to unseen data. This is consistent with expectations — Random Forest's ensemble averaging reduces the variance that causes single Decision Trees to overfit.

### 12.2 How strongly does price predict sales?
The correlation between price and estimated owners was [VALUE] across datasets. The feature importance plots show that Price accounts for approximately [X]% of the Random Forest's total importance, ranking [Nth] overall. This confirms that price is a meaningful but not dominant predictor — game quality (Positive_Ratio) and genre also play significant roles.

### 12.3 Is there a seasonal effect?
The t-test comparing holiday vs non-holiday releases showed [significant/no significant] difference (p = [VALUE]). Games released in October–December showed [higher/similar] median ownership, consistent with the hypothesis that holiday window releases benefit from increased consumer spending.

### 12.4 What is the optimal price point?
The price optimisation analysis suggests an optimal price of approximately \$[VALUE] (averaged across models and datasets). The current median price in the dataset is \$[ACTUAL].

### 12.5 Are games overpriced?
[Fill in based on opt_df output]  
Based on the model's predictions, games appear to be [overpriced by / underpriced by / priced near] \$[X] relative to the sales-maximising optimum. This suggests that [publishers are pricing above the demand curve's peak / the market could support slightly higher prices].

### 12.6 Limitations
- Steam data is PC-centric and does not represent console pricing dynamics
- Estimated owners is a proxy derived from SteamSpy ranges, not exact sales figures
- The model cannot account for brand recognition, marketing spend, or franchise history
- Price optimisation assumes other features are fixed — in practice, a $70 game may have higher production quality that also increases Positive_Ratio

### 12.7 Stakeholder Takeaway
Our analysis suggests that the sales-maximising price point for the average Steam game is approximately \$[X]. Publishers pricing significantly above this threshold can expect a measurable reduction in player reach. For independent developers especially, pricing at or below \$[X] may dramatically increase their player base while having a modest impact on per-unit revenue.


In [ ]:
# Final summary printout
print("="*65)
print("FINAL RESULTS SUMMARY")
print("="*65)

print("\n── Model Performance ──")
rmse_piv = summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE')
display(rmse_piv.round(4))

print("\n── Price Optimisation ──")
display(opt_df[['Model', 'Dataset',
                'Optimal_Price', 'Actual_Median_Price',
                'Overpriced']].to_string(index=False))

print("\n── Best model per dataset ──")
for col in rmse_piv.columns:
    best = rmse_piv[col].idxmin()
    val  = rmse_piv[col].min()
    print(f"  {col}: {best}  (RMSE = {val:.4f})")

print("\nAll plots saved to:", PLOT_DIR)